> [!NOTE]
> **Historical Exploration Notebook:** This notebook contains exploratory templates from earlier project milestones. The canonical, reproducible, and leakage-free controlled multi-model comparison (Logistic Regression, Decision Tree, Random Forest, Majority Baseline) is executed in [07_evaluation_comparison.ipynb](07_evaluation_comparison.ipynb).


# Decision Tree — interpretable non-linear model

## Plan

1. **Existing notebook.** `05_decision_tree.ipynb` was empty (0 bytes), so this notebook is new.
2. **Reuse of the LR decisions.** These are identical to LR, through the same `src/features.py` code:
   - the same 41 raw features
   - the same `FeatureCleaner` (`num_outbound_cmds` dropped, `su_attempted` 2 → 1)
   - the same `CorrelationFilter(0.98)`, which drops `num_root`, `srv_serror_rate`, `srv_rerror_rate` and `dst_host_srv_serror_rate`
   - the same OHE with `min_frequency=50`
   - the same label mapping, the same `train_sub`/`val` indices (`results/splits/`), the same CV folds (seed 42) and the same untouched test set
3. **Model-specific pipeline.** The `tree` family skips `log1p` and `StandardScaler`. Split thresholds are invariant to monotone transforms, so they add nothing; an ablation below verifies this.
4. **Search space (Optuna TPE, 30 trials, the same budget as LR):**
   - `criterion` ∈ {gini, entropy}
   - `max_depth` ∈ [3, 30]
   - `min_samples_split` ∈ [2, 200] (log)
   - `min_samples_leaf` ∈ [1, 100] (log)
   - `max_leaf_nodes` ∈ {None, 32 … 1024}
   - `ccp_alpha` ∈ [1e-6, 1e-2] (log)
   - `class_weight` ∈ {None, balanced}
   - `random_state = 42`
5. **Strategy.** Stratified 5-fold CV macro-F1 on `train_sub`, threshold on `val`, refit on KDDTrain+, test once. Training-vs-validation curves check overfitting.
6. **Visuals.** Optuna history, train/val/test bars, confusion/ROC/PR, depth and leaf-size curves, the pruning path, impurity vs permutation importance, the top of the tree and error analysis.
7. **Comparisons.**
   - Gini vs entropy
   - unrestricted vs tuned/pruned
   - depth and minimum leaf size
   - tree with vs without the linear transforms
   - LR on identical splits and metrics

In [ ]:
import sys, json, time
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))
from src import experiment as ex, plots
from src.features import build_pipeline, build_preprocessor
from src.evaluation import compute_metrics
from sklearn.tree import DecisionTreeClassifier, plot_tree, export_text
from sklearn.model_selection import cross_val_score
from sklearn.metrics import f1_score

rng = ex.init()
SEED, CFG = ex.SEED, ex.CFG
sns.set_theme(style="whitegrid")
MODEL = "decision_tree"
PREP = dict(corr_threshold=CFG["features"]["corr_threshold"], min_frequency=CFG["features"]["min_frequency"])

train, test, novel = ex.load_data()
idx_tr, idx_va = ex.get_splits(train)
X_all, y_all = ex.xy(train)
X_tr, y_tr, X_va, y_va = X_all.iloc[idx_tr], y_all[idx_tr], X_all.iloc[idx_va], y_all[idx_va]
run_id = ex.start_run(MODEL, {"family": "tree", "log_transform": False, **PREP})
print("MLflow run:", run_id, "| trials:", ex.n_trials())

In [ ]:
def make_pipeline(p, family="tree", **prep_over):
    est = DecisionTreeClassifier(criterion=p["criterion"], max_depth=p["max_depth"],
                                 min_samples_split=p["min_samples_split"], min_samples_leaf=p["min_samples_leaf"],
                                 max_leaf_nodes=p["max_leaf_nodes"], ccp_alpha=p["ccp_alpha"],
                                 class_weight=p["class_weight"], random_state=SEED)
    return build_pipeline(est, family, **{**PREP, **prep_over})

def suggest(trial):
    return {"criterion": trial.suggest_categorical("criterion", ["gini", "entropy"]),
            "max_depth": trial.suggest_int("max_depth", 3, 30),
            "min_samples_split": trial.suggest_int("min_samples_split", 2, 200, log=True),
            "min_samples_leaf": trial.suggest_int("min_samples_leaf", 1, 100, log=True),
            "max_leaf_nodes": trial.suggest_categorical("max_leaf_nodes", [None, 32, 64, 128, 256, 512, 1024]),
            "ccp_alpha": trial.suggest_float("ccp_alpha", 1e-6, 1e-2, log=True),
            "class_weight": trial.suggest_categorical("class_weight", [None, "balanced"])}

t0 = time.perf_counter()
study = ex.tune(MODEL, run_id, make_pipeline, suggest, X_tr, y_tr, cv_jobs=5)
tuning_seconds = time.perf_counter() - t0
best = study.best_params
print(f"best CV macro-F1 = {study.best_value:.4f} with {best}  ({tuning_seconds/60:.1f} min)")
fig = plots.optuna_overview(study, "ccp_alpha", hue="criterion"); ex.log_figure(run_id, fig, "optuna", MODEL); plt.show()

## 1. Train / validation diagnostics, threshold and the single test evaluation

In [ ]:
res = ex.fit_and_evaluate(MODEL, run_id, make_pipeline, best, train, test, idx_tr, idx_va, novel,
                          extra={"tuning": {"best_cv_f1_macro": study.best_value, "n_trials": len(study.trials),
                                            "tuning_seconds": tuning_seconds}})
pd.DataFrame({k: res["metrics"][k] for k in ["train", "val", "test", "test_at_0.5"]}).T[
    ["threshold", "accuracy", "precision", "recall", "f1", "f1_macro", "roc_auc", "pr_auc", "fpr", "fnr"]].round(4)

In [ ]:
fig = plots.metric_bars(res["metrics"], "Decision Tree: train vs validation vs test"); ex.log_figure(run_id, fig, "metric_bars", MODEL); plt.show()
fig = plots.confusion_and_curves(res["y_va"], res["p_va"], res["y_te"], res["p_te"], res["threshold"], "DT")
ex.log_figure(run_id, fig, "confusion_roc_pr_threshold", MODEL); plt.show()

## 2. Complexity, overfitting and pruning (fit on `train_sub`, score on `train_sub` and `val`)

In [ ]:
prep = build_preprocessor("tree", **PREP).fit(X_tr)
Ztr, Zva = prep.transform(X_tr), prep.transform(X_va)
base = {k: v for k, v in best.items()}

def tv(**over):
    p = {**base, **over}
    m = DecisionTreeClassifier(criterion=p["criterion"], max_depth=p["max_depth"], min_samples_split=p["min_samples_split"],
                               min_samples_leaf=p["min_samples_leaf"], max_leaf_nodes=p["max_leaf_nodes"],
                               ccp_alpha=p["ccp_alpha"], class_weight=p["class_weight"], random_state=SEED).fit(Ztr, y_tr)
    return (f1_score(y_tr, m.predict(Ztr), average="macro"), f1_score(y_va, m.predict(Zva), average="macro"), m.get_n_leaves())

free = dict(max_depth=None, min_samples_split=2, min_samples_leaf=1, max_leaf_nodes=None, ccp_alpha=0.0)
depths = list(range(1, 31))
dcurve = pd.DataFrame([tv(**{**free, "max_depth": d}) for d in depths], index=depths, columns=["train", "val", "leaves"])
leafs = [1, 2, 3, 5, 8, 12, 20, 30, 50, 80, 120, 200, 300]
lcurve = pd.DataFrame([tv(**{**free, "min_samples_leaf": l}) for l in leafs], index=leafs, columns=["train", "val", "leaves"])

path = DecisionTreeClassifier(criterion=best["criterion"], random_state=SEED).cost_complexity_pruning_path(Ztr, y_tr)
alphas = np.unique(np.quantile(path.ccp_alphas[path.ccp_alphas > 0], np.linspace(0, .995, 30)))
acurve = pd.DataFrame([tv(**{**free, "ccp_alpha": a}) for a in alphas], index=alphas, columns=["train", "val", "leaves"])

fig, axes = plt.subplots(1, 3, figsize=(20, 4.6))
for ax, cur, xl, logx in [(axes[0], dcurve, "max_depth (other limits off)", False),
                          (axes[1], lcurve, "min_samples_leaf (other limits off)", True),
                          (axes[2], acurve, "ccp_alpha (pruning path)", True)]:
    ax.plot(cur.index, cur["train"], "o-", ms=3, label="train_sub"); ax.plot(cur.index, cur["val"], "s-", ms=3, label="val")
    ax2 = ax.twinx(); ax2.plot(cur.index, cur["leaves"], c="#8C8C8C", ls=":", label="leaves"); ax2.set_yscale("log"); ax2.set_ylabel("leaves")
    ax2.grid(False)
    if logx: ax.set_xscale("log")
    ax.set_xlabel(xl); ax.set_ylabel("macro-F1"); ax.set_ylim(max(.8, cur[["train", "val"]].values.min() - .01), 1.001); ax.legend(loc="lower right")
axes[0].set_title("Depth: training vs validation"); axes[1].set_title("Minimum leaf size"); axes[2].set_title("Cost-complexity pruning")
fig.tight_layout(); ex.log_figure(run_id, fig, "complexity_curves", MODEL); plt.show()
print("best val depth:", dcurve["val"].idxmax(), "| best val min_samples_leaf:", lcurve["val"].idxmax(),
      "| best val ccp_alpha: %.2e" % acurve["val"].idxmax())

## 3. Experiments: Gini vs Entropy, unrestricted vs tuned, and the effect of linear transforms

In [ ]:
def cv(pipe):
    s = cross_val_score(pipe, X_tr, y_tr, cv=ex.cv_splitter(), scoring="f1_macro", n_jobs=5)
    return s.mean(), s.std()

other = "entropy" if best["criterion"] == "gini" else "gini"
exps = {
    f"tuned ({best['criterion']})": make_pipeline(best),
    f"tuned params, {other}": make_pipeline({**best, "criterion": other}),
    "unrestricted (gini)": make_pipeline({**best, **free, "criterion": "gini", "class_weight": None}),
    "unrestricted (entropy)": make_pipeline({**best, **free, "criterion": "entropy", "class_weight": None}),
    "tuned + log1p/scaling (linear prep)": make_pipeline(best, family="linear"),
    "tuned, no correlation filter": make_pipeline(best, corr_threshold=None),
}
exp = pd.DataFrame({k: cv(v) for k, v in exps.items()}, index=["cv_f1_macro", "std"]).T
tt = {}
for name, p in [("unrestricted", {**best, **free, "criterion": "gini", "class_weight": None}), ("tuned/pruned", best)]:
    m = make_pipeline(p).fit(X_tr, y_tr)
    tt[name] = {"train_sub": f1_score(y_tr, m.predict(X_tr), average="macro"), "val": f1_score(y_va, m.predict(X_va), average="macro"),
                "leaves": m[-1].get_n_leaves(), "depth": m[-1].get_depth()}
tt = pd.DataFrame(tt).T

fig, axes = plt.subplots(1, 2, figsize=(17, 4.2), gridspec_kw={"width_ratios": [1.8, 1]})
e = exp.sort_values("cv_f1_macro")
axes[0].barh(e.index, e["cv_f1_macro"], xerr=e["std"], color=["#C44E52" if i.startswith("tuned (") else "#8C8C8C" for i in e.index])
axes[0].set_xlim(e["cv_f1_macro"].min() - .003, e["cv_f1_macro"].max() + .002); axes[0].set_title("CV macro-F1 (5-fold, train_sub)")
tt[["train_sub", "val"]].plot(kind="bar", ax=axes[1], rot=0, color=["#4C72B0", "#55A868"])
axes[1].set_ylim(tt[["train_sub", "val"]].values.min() - .01, 1.002)
for i, (l, d) in enumerate(zip(tt["leaves"], tt["depth"])): axes[1].text(i, 1.0, f"{l} leaves, depth {d}", ha="center", fontsize=8)
axes[1].set_title("Overfitting: unrestricted vs tuned")
fig.tight_layout(); ex.log_figure(run_id, fig, "experiments", MODEL); plt.show()
exp.to_csv(PROJECT_ROOT / "results" / "tuning" / f"{MODEL}_experiments.csv")
import mlflow
ex.setup_mlflow()
with mlflow.start_run(run_id=run_id):
    mlflow.log_metrics({f"exp_{i}": v for i, v in enumerate(exp["cv_f1_macro"])})
    mlflow.log_dict(exp.round(5).to_dict(), "experiments/cv_experiments.json")
display(exp.round(4)); tt.round(4)

## 4. Interpretability: feature importance and tree structure

In [ ]:
from sklearn.inspection import permutation_importance
final = res["final"]
names = np.array([n.split("__", 1)[1] for n in final[:-1].get_feature_names_out()])
imp = pd.Series(final[-1].feature_importances_, index=names).sort_values(ascending=False)
# permutation importance of the dev model on validation rows (raw features, whole pipeline)
pi = permutation_importance(res["dev"], X_va, y_va, scoring="f1_macro", n_repeats=5, random_state=SEED, n_jobs=5)
pimp = pd.Series(pi.importances_mean, index=X_va.columns).sort_values(ascending=False)
fig, axes = plt.subplots(1, 2, figsize=(16, 5.5))
t = imp.head(15); axes[0].barh(t.index[::-1], t.values[::-1], color="#55A868"); axes[0].set_title("Impurity importance (final tree)")
t = pimp.head(15); axes[1].barh(t.index[::-1], t.values[::-1], color="#8172B2"); axes[1].set_title("Permutation importance on val (raw features, Δ macro-F1)")
fig.tight_layout(); ex.log_figure(run_id, fig, "feature_importance", MODEL); plt.show()
imp.to_csv(PROJECT_ROOT / "results" / "tuning" / f"{MODEL}_feature_importance.csv")

In [ ]:
fig, ax = plt.subplots(figsize=(22, 8))
plot_tree(final[-1], max_depth=3, feature_names=list(names), class_names=["normal", "attack"], filled=True,
          impurity=False, proportion=True, rounded=True, fontsize=8, ax=ax)
ax.set_title(f"Final tree — top 4 levels of {final[-1].get_depth()} (leaves: {final[-1].get_n_leaves()})")
ex.log_figure(run_id, fig, "tree_top_levels", MODEL); plt.show()
rules = export_text(final[-1], feature_names=list(names), max_depth=6, decimals=3)
(PROJECT_ROOT / "results" / "tuning" / f"{MODEL}_rules.txt").write_text(rules)
print("\n".join(rules.splitlines()[:40]))

## 5. Error analysis

In [ ]:
fig = plots.error_analysis(res["preds"], test, "DT"); ex.log_figure(run_id, fig, "error_analysis", MODEL); plt.show()
fig = plots.missed_attack_types(res["preds"]); ex.log_figure(run_id, fig, "missed_attack_types", MODEL); plt.show()

## 6. Comparison with Logistic Regression (identical splits, folds and metrics)

In [ ]:
lr = json.loads((PROJECT_ROOT / "results" / "metrics" / "logistic_regression.json").read_text())
dt = res["payload"]
keys = ["f1_macro", "recall", "fpr", "roc_auc", "pr_auc"]
rows = []
for name, P in [("Logistic Regression", lr), ("Decision Tree", dt)]:
    for split in ["val", "test"]:
        rows += [{"model": name, "split": split, "metric": k, "value": P["metrics"][split][k]} for k in keys]
cmp = pd.DataFrame(rows)
g = sns.catplot(data=cmp, x="metric", y="value", hue="model", col="split", kind="bar", height=4, aspect=1.6)
for ax in g.axes.ravel():
    for c in ax.containers: ax.bar_label(c, fmt="%.3f", fontsize=7)
g.fig.suptitle("LR vs DT on identical data", y=1.03)
ex.log_figure(run_id, g.fig, "lr_vs_dt", MODEL); plt.show()
cmp.pivot_table(index=["split", "metric"], columns="model", values="value").round(4)

## 7. Summary

In [ ]:
m = res["metrics"]
pd.DataFrame({"value": [str(best), f"{study.best_value:.4f}", f"{res['threshold']:.2f}",
                        f"{m['train']['f1_macro']:.4f} / {m['val']['f1_macro']:.4f} / {m['test']['f1_macro']:.4f}",
                        f"{m['test']['recall']:.4f}", f"{m['test']['fpr']:.4f}", f"{m['test']['roc_auc']:.4f}",
                        f"{res['slices']['recall_novel']:.4f}", str(res["payload"]["complexity"]),
                        res["payload"]["artifacts"].get("onnx_status")]},
             index=["best params", "CV macro-F1", "threshold (val)", "macro-F1 train / val / test", "test attack recall",
                    "test FPR", "test ROC-AUC", "novel-attack recall", "complexity", "ONNX export"])